<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [21]</a>'.</span>

# Qwen3.5-27B QLoRA: DeltaNet rank sweep

Compare ranks **32, 40, 48, 56 and 64**, with **alpha = 2 × rank**, on NF4 `Qwen/Qwen3.5-27B` across seeds **42, 123 and 2026**.

Each seed uses one question split shared by all five ranks. Every run starts from a fresh base model and adapter; only rank and its paired alpha change. Targets include full-attention and MLP projections plus DeltaNet's `in_proj_qkv` and `out_proj`.

Run all cells from the top to train **15 adapters** and evaluate **15 adapters plus three frozen baselines**. Results include per-seed metrics, means, sample standard deviations and differences from the baseline. Checkpoints are selected by validation loss, independently of test metrics.


## Install dependencies

Use the Python 3.12 `qlora` kernel with `PYTHONNOUSERSITE=1`. Setup installs CUDA-enabled PyTorch and the training packages, then prints CUDA/compiler diagnostics. Full attention uses SDPA; DeltaNet uses the PyTorch fallback.

Defaults use `torch==2.10.0+cu126`. For another CUDA toolkit, set `QLORA_TORCH_PACKAGE` and `QLORA_TORCH_INDEX_URL` before setup. Restart the kernel after changing imported packages. The experiment records package versions and the model revision; use matching versions when comparing runs.


In [1]:
import os
import re
import shutil
import site
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, distribution, version
from pathlib import Path

print("Python:", sys.executable)
print("Environment:", sys.prefix)
# Require user-site isolation from kernel startup.
user_sites = site.getusersitepackages()
if isinstance(user_sites, str):
    user_sites = [user_sites]
if any(Path(entry).resolve() == Path(user_site).resolve()
       for entry in sys.path if entry for user_site in user_sites):
    raise RuntimeError(
        "This kernel includes ~/.local user-site packages. Register/select the "
        "qlora kernel with PYTHONNOUSERSITE=1, then restart it. See Training.md "
        "for kernel registration instructions."
    )
# Isolate installation subprocesses too.
os.environ["PYTHONNOUSERSITE"] = "1"
# Override both settings for a different CUDA toolkit.
torch_package = os.environ.get("QLORA_TORCH_PACKAGE", "torch==2.10.0+cu126")
torch_index = os.environ.get("QLORA_TORCH_INDEX_URL", "https://download.pytorch.org/whl/cu126")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-user", torch_package,
     "--index-url", torch_index], check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-user", "-U", "ipywidgets", torch_package,
     "transformers>=5.16.1", "torchvision", "kernels", "accelerate", "peft", "datasets", "bitsandbytes", "trl",
     "sentencepiece", "numpy>=2,<3", "pandas", "pyarrow", "scipy", "scikit-learn",
     "numexpr>=2.10.2", "bottleneck>=1.4.2", "ninja", "packaging",
     "huggingface_hub", "python-dotenv", "setuptools", "wheel"],
    check=True,
)

# Find an available CUDA compiler.
nvcc_candidates = []
for variable in ("CUDA_HOME", "CUDA_PATH"):
    if os.environ.get(variable):
        nvcc_candidates.append(Path(os.environ[variable]) / "bin" / "nvcc")
if path_nvcc := shutil.which("nvcc"):
    nvcc_candidates.append(Path(path_nvcc))
for package in ("nvidia-cuda-nvcc", "nvidia-cuda-nvcc-cu12", "nvidia-cuda-nvcc-cu11"):
    try:
        nvcc_dist = distribution(package)
    except PackageNotFoundError:
        continue
    nvcc_candidates.extend(
        Path(nvcc_dist.locate_file(file)) for file in nvcc_dist.files or []
        if str(file).endswith("/bin/nvcc")
    )
nvcc = next((path.resolve() for path in nvcc_candidates
             if path.is_file() and os.access(path, os.X_OK)), None)
if nvcc is not None:
    cuda_home = nvcc.parent.parent
    os.environ["CUDA_HOME"] = str(cuda_home)
    os.environ["PATH"] = str(nvcc.parent) + os.pathsep + os.environ.get("PATH", "")
    os.environ["CPATH"] = str(cuda_home / "include") + os.pathsep + os.environ.get("CPATH", "")
    print("CUDA compiler:", nvcc)
else:
    print("nvcc not found in this kernel. "
          "Training can use SDPA. To build optional extensions, "
          "load a matching cluster CUDA module before starting the notebook job.")

os.environ.setdefault("MAX_JOBS", "1")
# Let PyTorch infer GPU architecture unless TORCH_CUDA_ARCH_LIST was supplied.
system_cc = os.environ.get("CC") or shutil.which("gcc")
system_cxx = os.environ.get("CXX") or shutil.which("g++")
if system_cc:
    os.environ["CC"] = system_cc
if system_cxx:
    os.environ["CXX"] = system_cxx
if not system_cc or not system_cxx:
    print("Host compiler missing; load the cluster's GCC/G++ module before training.")

# A live kernel cannot safely switch to a different PyTorch binary.
if "torch" in sys.modules and sys.modules["torch"].__version__ != version("torch"):
    raise RuntimeError("PyTorch changed on disk. Restart the kernel and rerun setup.")

import torch

print("PyTorch:", torch.__version__, "| Built for CUDA:", torch.version.cuda,
      "| GPU available:", torch.cuda.is_available())
nvcc_version = None
if nvcc is not None:
    probe = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True, check=False)
    match = re.search(r"\brelease\s+(\d+\.\d+)", probe.stdout)
    if probe.returncode == 0 and match:
        nvcc_version = match.group(1)
    print("Compiler CUDA:", nvcc_version or "unknown")
    if nvcc_version is None or nvcc_version != torch.version.cuda:
        print(f"Compiler diagnostic: nvcc CUDA {nvcc_version or 'unknown'} "
              f"does not match PyTorch CUDA {torch.version.cuda}. "
              "Use matching toolkit/PyTorch versions to compile optional kernels.")

print("Qwen3.5-27B uses SDPA for full attention and the PyTorch fallback for DeltaNet linear attention.")


Python: /common/home/projectgrps/CS480/CS480G3/.conda/envs/qlora/bin/python
Environment: /common/home/projectgrps/CS480/CS480G3/.conda/envs/qlora


Looking in indexes: https://download.pytorch.org/whl/cu126


  Using cached torchvision-0.29.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.7 kB)


INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.29.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.7 kB)
  Using cached torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.6 kB)
  Using cached torchvision-0.27.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)


CUDA compiler: /opt/apps/software/CUDA/12.4.0/bin/nvcc


PyTorch: 2.10.0+cu126 | Built for CUDA: 12.6 | GPU available: True
Compiler CUDA: 12.4
Compiler diagnostic: nvcc CUDA 12.4 does not match PyTorch CUDA 12.6. Use matching toolkit/PyTorch versions to compile optional kernels.
Qwen3.5-27B uses SDPA for full attention and the PyTorch fallback for DeltaNet linear attention.


## Imports and hardware

Print CUDA availability, GPU memory and compiler details. Training requires one CUDA GPU. Optional `.env` settings are loaded; Hugging Face uses `HF_TOKEN` when supplied.


In [2]:
import ast
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# Set the compiler before importing PyTorch/Triton.
cc = os.environ.get("CC") or next(
    (path for name in ("gcc", "clang") if (path := shutil.which(name))),
    None,
)
if cc:
    os.environ["CC"] = cc

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForMultimodalLM,
    AutoProcessor,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

load_dotenv()

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")


PyTorch: 2.10.0+cu126 | CUDA: True | Compiler: /opt/apps/software/GCCcore/13.3.0/bin/gcc
GPU: NVIDIA L40S | VRAM: 44.4 GiB


## Configuration

- Sweep `(rank, alpha)` pairs `(32, 64)`, `(40, 80)`, `(48, 96)`, `(56, 112)` and `(64, 128)` for each seed.
- Keep learning rate `5e-5`, dropout `0.05`, weight decay `0.05` and effective batch size 8. Train for up to three epochs with validation-loss scheduling and early stopping.
- Use ordinary LoRA scaling (`use_rslora=False`), so `alpha/rank = 2` throughout.
- Set `RUN_TRAINING=False` for data/tokenizer checks only. `RUN_TEST_EVALUATION` controls held-out evaluation; upload is disabled by default.
- Save each sweep in a fresh output directory. Set `MODEL_REVISION` to a recorded commit to reproduce a checkpoint.


In [3]:
MODEL_ID = "Qwen/Qwen3.5-27B"
MODEL_REVISION = None  # Set to a recorded commit hash to reproduce an earlier run.
DATASET_PATH = next(
    (str(parent / "Datasets/training_data_moh_eng_rubrics.csv")
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Datasets/training_data_moh_eng_rubrics.csv").is_file()),
    "./Datasets/training_data_moh_eng_rubrics.csv",
)
OUTPUT_DIR = "./qwen35-27b-grading-qlora-inprojqkv-outproj-rank-multiseed"
RUN_TRAINING = True
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = False  # Opt in only when ready to publish one evaluated adapter.
UPLOAD_SEED = 42
UPLOAD_RANK = 48
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
evaluation_completed = False
TRAINING_SEEDS = (42, 123, 2026)
MAX_SEQ_LENGTH = None  # Auto-size from formatted training/validation data.

LORA_RANKS = (32, 40, 48, 56, 64)
if not LORA_RANKS or any(type(rank) is not int or rank <= 0 for rank in LORA_RANKS):
    raise ValueError("LORA_RANKS must contain positive integers.")
if len(set(LORA_RANKS)) != len(LORA_RANKS):
    raise ValueError("LORA_RANKS must contain unique ranks.")
if not TRAINING_SEEDS or len(set(TRAINING_SEEDS)) != len(TRAINING_SEEDS):
    raise ValueError("TRAINING_SEEDS must be nonempty and contain unique seeds.")
# Use the same nine projection targets for every rank.
LORA_TARGET_SUFFIXES = (
    "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "out_proj",
)
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size 8 on one GPU.
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKEN_BUDGET = 128
GENERATION_TOKENS_PER_CRITERION = 8

set_seed(TRAINING_SEEDS[0])


## Load the dataset

Load the rubric CSV and preview its rows. Require prompt fields, criterion labels/maxima and final scores. Preserve blank student answers and literal `NA` text.


In [4]:
required_columns = [
    "question", "desired_answer", "student_answer", "criteria[]",
    "criteria_score[]", "criteria_max_score[]", "student_score",
]
if not Path(DATASET_PATH).is_file():
    raise FileNotFoundError(f"Rubric dataset not found: {DATASET_PATH}. Set DATASET_PATH in Configuration.")
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())


Rows: 3778


,id,question,desired_answer,student_answer,criteria[],criteria_score[],criteria_max_score[],student_score
0,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,High risk problems are address in the prototyp...,"[""Answer mentions simulating behaviour of port...",[0],[1],0
1,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,To simulate portions of the desired final prod...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
2,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,A prototype program simulates the behaviors of...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
3,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,Defined in the Specification phase a prototype...,"[""Answer mentions simulating behaviour of port...",[1],[1],1
4,1.1,What is the role of a prototype program in pro...,To simulate the behaviour of portions of the d...,It is used to let the users have a first idea ...,"[""Answer mentions simulating behaviour of port...",[0],[1],0


## Parse and validate rows

Parse JSON/Python arrays and legacy bracket arrays, preserving commas inside criteria. Require nonempty criteria, aligned binary labels and finite nonnegative score metadata. Report differences between recorded final scores and criterion totals; evaluation uses the recorded scores.


In [5]:
def parse_array(value, expected_count=None):
    """Parse structured arrays or legacy bracket text without losing criterion commas."""
    items = value if isinstance(value, list) else None
    if items is None:
        if not isinstance(value, str):
            raise TypeError(f"Expected an array or string, got {type(value).__name__}.")
        text = value.strip()
        # Unwrap quoted JSON/Python strings before handling legacy unquoted entries.
        for _ in range(3):
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                try:
                    parsed = ast.literal_eval(text)
                except (ValueError, SyntaxError):
                    break
            if isinstance(parsed, list):
                items = parsed
                break
            if not isinstance(parsed, str):
                raise ValueError(f"Expected a list, got {value!r}.")
            text = parsed.strip()
        if items is None:
            if not (text.startswith("[") and text.endswith("]")):
                raise ValueError(f"Expected a bracketed list, got {value!r}.")
            inner = text[1:-1].strip()
            if not inner:
                items = []
            elif expected_count == 1:
                items = [inner]
            elif re.search(r",\s*(?=Answer\s)", inner):
                items = re.split(r",\s*(?=Answer\s)", inner)
            else:
                items = next(csv.reader([inner], skipinitialspace=True))
            items = [item.strip() for item in items]
    if expected_count is not None and len(items) != expected_count:
        raise ValueError(f"Expected {expected_count} entries, parsed {len(items)}: {value!r}")
    return items


def parse_binary_scores(value):
    values = parse_array(value)
    if any(type(x) not in (int, str) or str(x) not in ("0", "1") for x in values):
        raise ValueError(f"Expected binary integers, got {value!r}")
    return [int(x) for x in values]


def parse_score_metadata(value):
    values = parse_array(value)
    if any(isinstance(x, bool) for x in values):
        raise ValueError("Criterion maxima cannot be booleans.")
    return [float(x) for x in values]


records = []
errors = []
for index, row in enumerate(df.to_dict("records")):
    try:
        scores = parse_binary_scores(row["criteria_score[]"])
        criteria = parse_array(row["criteria[]"], expected_count=len(scores))
        maxima = parse_score_metadata(row["criteria_max_score[]"])
        final_score = float(row["student_score"])
        if not criteria or any(not isinstance(c, str) or not c.strip() for c in criteria):
            raise ValueError("Criteria must be nonempty strings.")
        if len(criteria) != len(maxima):
            raise ValueError("Criterion, label, and maximum-score counts must match.")
        if any(not math.isfinite(x) or x < 0 for x in [*maxima, final_score]):
            raise ValueError("Score metadata must be finite and nonnegative.")
        if not str(row["question"]).strip() or not str(row["desired_answer"]).strip():
            raise ValueError("Question and desired answer must be nonempty.")
        records.append({
            "row_index": index,
            **{key: str(row[key]) for key in ("question", "desired_answer", "student_answer")},
            "criteria": criteria, "criteria_score": scores,
            "criteria_max_score": maxima, "student_score": final_score,
        })
    except (ValueError, TypeError, SyntaxError) as error:
        errors.append(f"Row {index}: {error}")
if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))
print(f"All {len(records)} rows passed validation.")
score_disagreements = sum(
    not math.isclose(
        sum(score * maximum for score, maximum in zip(row["criteria_score"], row["criteria_max_score"])),
        row["student_score"],
    )
    for row in records
)
print("Recorded final scores differing from criterion totals:", score_disagreements)


All 3778 rows passed validation.
Recorded final scores differing from criterion totals: 0


## Grading prompt

Use one prompt for training and inference. Request only an ordered JSON array of binary criterion decisions.


In [6]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)


You are a strict grading assistant.

Your task is to evaluate whether a student's answer fulfills each grading criterion.

For each criterion:
- Output 1 if the criterion is fulfilled.
- Output 0 if the criterion is not fulfilled.

Evaluate each criterion independently.

Return ONLY a valid JSON array containing 0 or 1.
The output array must contain exactly one value for each criterion, in the same order as the criteria provided.

Do not include explanations.
Do not include markdown.
Do not include any text before or after the JSON array.


## Build examples

Keep prompts and assistant completions separate. Retain structured fields for evaluation.


In [7]:
def make_prompt(question, desired_answer, student_answer, criteria):
    """Build the shared grading prompt without score metadata."""
    criteria_text = "\n".join(f"{i}. {text}" for i, text in enumerate(criteria, 1))
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]


dataset = Dataset.from_list([
    {
        **row,
        "prompt": make_prompt(row["question"], row["desired_answer"], row["student_answer"], row["criteria"]),
        "completion": [{"role": "assistant", "content": json.dumps(row["criteria_score"])}],
    }
    for row in records
])
print(dataset[0]["prompt"])


[{'role': 'system', 'content': "You are a strict grading assistant.\n\nYour task is to evaluate whether a student's answer fulfills each grading criterion.\n\nFor each criterion:\n- Output 1 if the criterion is fulfilled.\n- Output 0 if the criterion is not fulfilled.\n\nEvaluate each criterion independently.\n\nReturn ONLY a valid JSON array containing 0 or 1.\nThe output array must contain exactly one value for each criterion, in the same order as the criteria provided.\n\nDo not include explanations.\nDo not include markdown.\nDo not include any text before or after the JSON array."}, {'role': 'user', 'content': 'Question:\nWhat is the role of a prototype program in problem solving?\n\nDesired answer:\nTo simulate the behaviour of portions of the desired software product.\n\nStudent answer:\nHigh risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.\n

## Split by question for each seed

Split unique stripped questions approximately 80/10/10 into training, validation and test sets. Require disjoint, nonempty splits covering all rows. Preserve seeded question order and reuse each seed's split for every rank and its baseline. Held-out questions change across seeds.


In [8]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))


def make_question_splits(split_seed):
    shuffled_questions = unique_questions.copy()
    random.Random(split_seed).shuffle(shuffled_questions)
    n_questions = len(shuffled_questions)
    train_end = int(n_questions * 0.80)
    validation_end = train_end + int(n_questions * 0.10)
    question_splits = {
        "train": shuffled_questions[:train_end],
        "validation": shuffled_questions[train_end:validation_end],
        "test": shuffled_questions[validation_end:],
    }
    if any(not questions for questions in question_splits.values()):
        raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
    assert sum(map(len, question_splits.values())) == len(set.union(
        *(set(questions) for questions in question_splits.values())
    )) == n_questions
    indices_by_question = defaultdict(list)
    for index, question in enumerate(question_keys):
        indices_by_question[question].append(index)
    # Preserve the seeded question order in training, validation, and test.
    splits = {
        name: dataset.select([
            index for question in questions for index in indices_by_question[question]
        ])
        for name, questions in question_splits.items()
    }
    assert sum(map(len, splits.values())) == len(dataset)
    return {"split_seed": split_seed, "question_splits": question_splits, "splits": splits}


seed_data = {seed: make_question_splits(seed) for seed in TRAINING_SEEDS}
for seed, data in seed_data.items():
    print(f"Seed {seed} question split:")
    for name, split in data["splits"].items():
        print(f"  {name}: {len(split)} rows / {len(data['question_splits'][name])} questions")


Seed 42 question split:
  train: 2980 rows / 76 questions
  validation: 409 rows / 9 questions
  test: 389 rows / 10 questions
Seed 123 question split:
  train: 2889 rows / 76 questions
  validation: 611 rows / 9 questions
  test: 278 rows / 10 questions
Seed 2026 question split:
  train: 3113 rows / 76 questions
  validation: 269 rows / 9 questions
  test: 396 rows / 10 questions


## Tokenizer

Load `AutoProcessor` and its text tokenizer. Share the native chat template across training and inference, with thinking disabled. Use and record the tokenizer's resolved checkpoint revision when available.


In [9]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = getattr(processor, "tokenizer", processor)
if getattr(processor, "chat_template", None):
    tokenizer.chat_template = processor.chat_template
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# Prefer the resolved commit hash when the tokenizer provides it.
MODEL_REVISION = tokenizer.init_kwargs.get("_commit_hash") or MODEL_REVISION
print("Tokenizer:", tokenizer.__class__.__name__)


Tokenizer: Qwen3_5Tokenizer


## Tokenize and verify completion masks

Prepare each seed's training/validation examples once for all ranks. Verify that generation prompts exactly prefix full conversations; supervise completion tokens only. Size the sequence limit from training/validation lengths, or reject examples exceeding an explicit cap. Test examples do not set training limits.


In [10]:
def tokenize_example(example):
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


for seed, data in seed_data.items():
    formatted_train, formatted_validation = (
        data["splits"][name].map(
            tokenize_example, remove_columns=data["splits"][name].column_names,
            desc=f"Seed {seed}: tokenize {name}",
        )
        for name in ("train", "validation")
    )
    lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
    measured_max_length = max(lengths)
    sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
    if sequence_limit < measured_max_length:
        raise ValueError(
            f"Seed {seed}: {sum(length > sequence_limit for length in lengths)} examples exceed "
            f"{sequence_limit} tokens; longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
        )
    data.update({
        "formatted_train": formatted_train, "formatted_validation": formatted_validation,
        "sequence_limit": sequence_limit,
    })
    print(f"Seed {seed} training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
    print("Sequence limit:", sequence_limit)
    print("Example supervised completion:", tokenizer.decode(
        [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
    ))


Seed 42: tokenize train:   0%|          | 0/2980 [00:00<?, ? examples/s]

Seed 42: tokenize validation:   0%|          | 0/409 [00:00<?, ? examples/s]

Seed 42 training/validation token lengths: {'count': 3389.0, 'mean': 296.1, 'std': 108.5, 'min': 178.0, '25%': 221.0, '50%': 258.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 123: tokenize train:   0%|          | 0/2889 [00:00<?, ? examples/s]

Seed 123: tokenize validation:   0%|          | 0/611 [00:00<?, ? examples/s]

Seed 123 training/validation token lengths: {'count': 3500.0, 'mean': 295.5, 'std': 105.2, 'min': 178.0, '25%': 222.0, '50%': 266.0, '75%': 327.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 1, 1]<|im_end|>



Seed 2026: tokenize train:   0%|          | 0/3113 [00:00<?, ? examples/s]

Seed 2026: tokenize validation:   0%|          | 0/269 [00:00<?, ? examples/s]

Seed 2026 training/validation token lengths: {'count': 3382.0, 'mean': 296.6, 'std': 108.3, 'min': 179.0, '25%': 221.0, '50%': 259.0, '75%': 331.0, 'max': 806.0}
Sequence limit: 832
Example supervised completion: [1, 0]<|im_end|>



## Quantisation and model loading

Load a fresh text-only `AutoModelForMultimodalLM` for every seed/rank pair. Use NF4 double quantisation, BF16 when supported (otherwise FP16), SDPA and one GPU. Disable caching during training and release each model before the next run.


In [11]:
def load_quantized_base_model():
    """Load a fresh Qwen3.5-27B checkpoint in NF4 on one GPU."""
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    if not cc:
        raise RuntimeError("Install a C compiler with the dependency cell, then restart the kernel.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook's batch and validation schedule assume one training GPU.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=bnb_config, device_map={"": 0},
        dtype=compute_dtype, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.config.get_text_config().use_cache = False
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print("Loaded in 4-bit:", model.is_loaded_in_4bit, "| Compute dtype:", compute_dtype)
    return model, compute_dtype


## Attach LoRA adapters

Target `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`, `in_proj_qkv` and `out_proj` under the language decoder. Require both DeltaNet projections in every linear-attention block and print target counts and trainable parameters.

Use the current rank and `alpha = 2 × rank`, dropout `0.05` and no bias training. Freeze base weights and untargeted modules.


In [12]:
def attach_fresh_lora(model, rank, alpha):
    """Attach the current rank to the attention, MLP and DeltaNet targets."""
    for param in model.parameters():
        param.requires_grad_(False)
    candidate_suffixes = set(LORA_TARGET_SUFFIXES)
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.language_model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in candidate_suffixes
    ]
    if not target_modules:
        raise RuntimeError("No Qwen3.5-27B language projection targets found under model.language_model.layers; inspect model.named_modules().")
    # Require QKV and output targets in every DeltaNet block.
    deltanet_layers = [
        name for name, _ in model.named_modules()
        if name.startswith("model.language_model.layers.") and name.endswith(".linear_attn")
    ]
    required_targets = {
        f"{name}.{suffix}" for name in deltanet_layers
        for suffix in ("in_proj_qkv", "out_proj")
    }
    missing_targets = sorted(required_targets - set(target_modules))
    if not deltanet_layers or missing_targets:
        raise RuntimeError(f"Missing Gated DeltaNet projection targets: {missing_targets or 'no linear_attn layers found'}")
    print("LoRA projection layers:", len(target_modules))
    print("Targets by projection:", {
        suffix: sum(name.rsplit(".", 1)[-1] == suffix for name in target_modules)
        for suffix in LORA_TARGET_SUFFIXES
    })
    model = get_peft_model(model, LoraConfig(
        r=rank, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
        use_rslora=False, bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()
    return model


## Training configuration

Reuse each seed's sequence limit and approximately four validation/checkpoint intervals per epoch for all ranks. Use paged 8-bit AdamW, mixed precision, gradient checkpointing and completion-only NLL without packing.

Reduce learning rate on validation-loss plateaus, stop after four checks without improvement and restore the lowest-loss checkpoint. Reset Trainer's `seed` and `data_seed` for every run. Test metrics do not select checkpoints.


In [13]:
for seed, data in seed_data.items():
    microbatches_per_epoch = math.ceil(len(data["formatted_train"]) / TRAIN_BATCH_SIZE)
    optimizer_steps_per_epoch = math.ceil(microbatches_per_epoch / GRADIENT_ACCUMULATION_STEPS)
    data["optimizer_steps_per_epoch"] = optimizer_steps_per_epoch
    data["quarter_epoch_steps"] = max(1, round(optimizer_steps_per_epoch / VALIDATIONS_PER_EPOCH))
    print(f"Seed {seed}: {optimizer_steps_per_epoch} optimizer steps per epoch; "
          f"evaluate/save every {data['quarter_epoch_steps']} optimizer steps")


def make_training_args(run_dir, compute_dtype, training_seed, data):
    """Build shared controls with this seed's data and independent run state."""
    quarter_epoch_steps = data["quarter_epoch_steps"]
    return SFTConfig(
        output_dir=str(run_dir),
        seed=training_seed,
        data_seed=training_seed,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        lr_scheduler_type="reduce_lr_on_plateau",
        lr_scheduler_kwargs={
            "mode": "min", "factor": LR_REDUCTION_FACTOR,
            "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
        },
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        activation_offloading=False,
        loss_type="nll",
        completion_only_loss=True,
        max_length=data["sequence_limit"],
        packing=False,
        padding_free=False,
        logging_steps=max(1, quarter_epoch_steps // 2),
        save_strategy="steps",
        save_steps=quarter_epoch_steps,
        eval_strategy="steps",
        eval_steps=quarter_epoch_steps,
        prediction_loss_only=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        report_to="none",
    )


Seed 42: 373 optimizer steps per epoch; evaluate/save every 93 optimizer steps
Seed 123: 362 optimizer steps per epoch; evaluate/save every 90 optimizer steps
Seed 2026: 390 optimizer steps per epoch; evaluate/save every 98 optimizer steps


## Create the trainer

Create fresh trainer, optimizer, scheduler and early-stopping state for every seed/rank pair. Reuse that seed's prepared examples; validation computes loss only.


In [14]:
def create_trainer(model, run_dir, compute_dtype, training_seed, data):
    """Create independent training state for the current adapter."""
    return SFTTrainer(
        model=model, args=make_training_args(run_dir, compute_dtype, training_seed, data),
        train_dataset=data["formatted_train"], eval_dataset=data["formatted_validation"],
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )],
    )


## Check the collated loss mask

Check up to two examples before each training run. Prompt and padding labels must be `-100`; completion tokens remain supervised.


In [15]:
def verify_completion_masks(trainer):
    """Verify that the collator supervises completion tokens only."""
    samples = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels = batch["labels"][index]
        length = len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to loss."
        assert (labels[:length] == -100).any() and (labels[:length] != -100).any()
    print("Prompt/completion/padding masks verified.")


## Train and save

Restore the lowest-validation-loss checkpoint and save the adapter, tokenizer, training metrics, loss history and checkpoint metadata in `seed_<seed>/rank_<rank>/alpha_<alpha>/`. Release training storage before inference.


In [16]:
def train_and_save(model, run_dir, compute_dtype, training_seed, data):
    """Save the validation-selected adapter and release training storage."""
    trainer = None
    try:
        trainer = create_trainer(model, run_dir, compute_dtype, training_seed, data)
        verify_completion_masks(trainer)
        train_result = trainer.train()
        trainer.save_model(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        (run_dir / "training_metrics.json").write_text(json.dumps(train_result.metrics, indent=2) + "\n")
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
        (run_dir / "best_checkpoint.json").write_text(json.dumps({
            "checkpoint": trainer.state.best_model_checkpoint,
            "validation_loss": trainer.state.best_metric,
        }, indent=2) + "\n")
        print("Saved best adapter and tokenizer to:", run_dir)
        display(train_result.metrics)
        return trainer.model
    finally:
        if trainer is not None:
            trainer.model.zero_grad(set_to_none=True)
            trainer.optimizer = None
            trainer.lr_scheduler = None
            trainer.accelerator.free_memory()
        trainer = None
        gc.collect()
        torch.cuda.empty_cache()


## Prepare inference

Disable gradient checkpointing, enable caching and switch the trained model to evaluation mode.


In [17]:
def prepare_inference(model):
    """Switch the trained model to cached inference."""
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.config.get_text_config().use_cache = True
    model.eval()


## Prediction helper

Generate greedily with thinking disabled and a token budget of `max(128, 8 × criterion count)`. Require integer binary arrays of the expected length. If JSON decoding fails, recover the last matching binary array and flag the output as recovered.


In [18]:
def parse_prediction(raw_output, expected_count):
    """Validate binary decisions, recovering an embedded array when needed."""
    recovered = False
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        scores = next((
            parsed for candidate in reversed(candidates)
            if len(parsed := json.loads(candidate)) == expected_count
        ), None)
        recovered = scores is not None
    error = None
    if not isinstance(scores, list):
        error = "Output was not a JSON array."
    elif len(scores) != expected_count:
        error = f"Expected {expected_count} scores, received {len(scores)}."
    elif any(type(score) is not int or score not in (0, 1) for score in scores):
        error = "Scores must be integer 0 or 1."
    return {
        "valid": error is None, "raw_output": raw_output,
        "scores": scores if error is None else None,
        "error": error, "recovered": recovered,
    }


@torch.inference_mode()
def predict_criteria_scores(question, desired_answer, student_answer, criteria, prediction_model):
    """Generate one greedy rubric prediction with the shared parser."""
    inputs = tokenizer.apply_chat_template(
        make_prompt(question, desired_answer, student_answer, criteria),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(prediction_model.get_input_embeddings().weight.device)
    generated = prediction_model.generate(
        **inputs, do_sample=False,
        max_new_tokens=max(GENERATION_MIN_TOKEN_BUDGET, len(criteria) * GENERATION_TOKENS_PER_CRITERION),
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_prediction(raw_output, len(criteria))


## Manual prediction

Print one illustrative prediction after each training run. Its expected array is `[1, 0, 0]`; it does not contribute to held-out metrics.


In [19]:
def show_manual_prediction(model):
    """Print an illustrative prediction outside the test-set metrics."""
    result = predict_criteria_scores(
        question="Explain why HTTPS is more secure than HTTP.",
        desired_answer="HTTPS encrypts network traffic using TLS.",
        student_answer="HTTPS protects transmitted data using encryption.",
        criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
        prediction_model=model,
    )
    print(result)


## Evaluate the held-out test set

Evaluate every rank and the frozen baseline on the same test rows within each seed. Compute final scores from predicted decisions × criterion maxima, compared with recorded `student_score`.

Invalid outputs fail exact-array and final-score accuracy. Criterion metrics, MAE, RMSE, Pearson and QWK use valid outputs only. Undefined metrics remain `NaN`. Save raw predictions, validity/recovery rates and per-example errors.


In [20]:
def evaluate_current_model(dataset, model_name, prediction_model):
    """Score one model state with the shared invalid-output and final-score rules."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    valid_true_final_scores = []
    valid_predicted_final_scores = []
    rows = []

    for example in dataset:
        true_scores = example["criteria_score"]
        max_scores = example["criteria_max_score"]
        true_final_score = example["student_score"]
        prediction = predict_criteria_scores(
            example["question"], example["desired_answer"],
            example["student_answer"], example["criteria"], prediction_model=prediction_model,
        )

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
            valid_true_final_scores.append(true_final_score)
            valid_predicted_final_scores.append(predicted_final_score)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "row_index": example["row_index"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    precision, recall, f1, _ = (
        precision_recall_fscore_support(criterion_true, criterion_pred, average="binary", zero_division=0)
        if criterion_true else (float("nan"),) * 4
    )
    if (
        len(valid_true_final_scores) > 1
        and len(set(valid_true_final_scores)) > 1
        and len(set(valid_predicted_final_scores)) > 1
    ):
        final_score_pearson = float(
            np.corrcoef(valid_true_final_scores, valid_predicted_final_scores)[0, 1]
        )
    else:
        final_score_pearson = float("nan")
    if valid_true_final_scores:
        final_score_qwk = float(cohen_kappa_score(
            valid_true_final_scores, valid_predicted_final_scores, weights="quadratic"
        ))
    else:
        final_score_qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": total - invalid_count,
        "recovered_output_rate": sum(row["recovered"] for row in rows) / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE, RMSE, Pearson, and QWK describe valid numeric predictions only.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
        "final_score_pearson": final_score_pearson,
        "final_score_qwk": final_score_qwk,
    }
    return metrics, rows


## Run the rank sweep

Train all five ranks for each seed, resetting the seed before every fresh model/adapter load. Evaluate the adapter-disabled baseline once per seed and each trained adapter once: **18 held-out evaluations**.

Record the dataset hash, splits, checkpoint revision, package versions and training settings in `experiment.json`. Save rank/alpha identifiers with metrics and predictions. Results are complete only when all configured evaluations cover their full test sets.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [21]:
def save_evaluation(run_dir, label, metrics, rows):
    """Persist complete metrics and raw predictions for one model state."""
    pd.DataFrame([metrics]).to_csv(run_dir / f"{label}_metrics.csv", index=False)
    pd.DataFrame(rows).to_json(run_dir / f"{label}_predictions.jsonl", orient="records", lines=True)


def configuration_label(rank):
    return f"QLoRA r={rank}, alpha={2 * rank}"


def configuration_dir(experiment_dir, seed, rank):
    return experiment_dir / f"seed_{seed}" / f"rank_{rank}" / f"alpha_{2 * rank}"


def run_configuration(rank, training_seed, experiment_dir, include_baseline=False):
    """Train and evaluate one seed/rank pair, then release GPU memory."""
    seed_dir = experiment_dir / f"seed_{training_seed}"
    seed_dir.mkdir(exist_ok=True)
    alpha = 2 * rank
    run_dir = configuration_dir(experiment_dir, training_seed, rank)
    data = seed_data[training_seed]
    test_dataset = data["splits"]["test"]
    run_dir.mkdir(parents=True, exist_ok=False)
    model = None
    results, predictions = [], []
    try:
        set_seed(training_seed)
        model, compute_dtype = load_quantized_base_model()
        model = attach_fresh_lora(model, rank, alpha)
        model = train_and_save(model, run_dir, compute_dtype, training_seed, data)
        prepare_inference(model)
        show_manual_prediction(model)
        if RUN_TEST_EVALUATION:
            if include_baseline:
                with model.disable_adapter():
                    metrics, rows = evaluate_current_model(test_dataset, "Baseline", prediction_model=model)
                metrics = {**metrics, "seed": training_seed, "rank": None, "alpha": None}
                rows = [{**row, "seed": training_seed, "rank": None, "alpha": None} for row in rows]
                save_evaluation(seed_dir, "baseline", metrics, rows)
                results.append(metrics)
                predictions.extend(rows)
            metrics, rows = evaluate_current_model(test_dataset, configuration_label(rank), prediction_model=model)
            metrics = {**metrics, "seed": training_seed, "rank": rank, "alpha": alpha}
            rows = [{**row, "seed": training_seed, "rank": rank, "alpha": alpha} for row in rows]
            save_evaluation(run_dir, "test", metrics, rows)
            results.append(metrics)
            predictions.extend(rows)
        return results, predictions
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()



def validate_seed_results(results_df, expected_seeds):
    """Require a baseline and every rank on each seed's complete test set."""
    expected_models = ["Baseline", *(configuration_label(rank) for rank in LORA_RANKS)]
    expected = {(seed, model) for seed in expected_seeds for model in expected_models}
    if results_df.empty or not {"seed", "rank", "alpha", "model", "examples"}.issubset(results_df.columns):
        raise ValueError("Seed evaluation results are empty or missing identifying columns.")
    if results_df.duplicated(["seed", "model"]).any():
        raise ValueError("Duplicate evaluation results for a seed/model pair.")
    actual = set(results_df[["seed", "model"]].itertuples(index=False, name=None))
    if actual != expected:
        raise ValueError(f"Incomplete results. Missing: {sorted(expected - actual)}; unexpected: {sorted(actual - expected)}")
    baseline = results_df.loc[results_df["model"] == "Baseline"]
    if not baseline[["rank", "alpha"]].isna().all().all():
        raise ValueError("Baseline rank and alpha must be empty.")
    for rank in LORA_RANKS:
        subset = results_df.loc[results_df["model"] == configuration_label(rank)]
        if not (subset["rank"].eq(rank).all() and subset["alpha"].eq(2 * rank).all()):
            raise ValueError(f"Incorrect rank/alpha identifiers for rank {rank}.")
    for seed in expected_seeds:
        subset = results_df.loc[results_df["seed"] == seed]
        if not subset["examples"].eq(len(seed_data[seed]["splits"]["test"])).all():
            raise ValueError(f"Seed {seed}: every model must evaluate the complete shared test set.")


all_metrics, all_prediction_rows = [], []
experiment_dir = None
evaluation_completed = False
if RUN_TRAINING:
    experiment_dir = Path(OUTPUT_DIR) / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S-%fZ")
    experiment_dir.mkdir(parents=True, exist_ok=False)
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    experiment_manifest = {
        "model": MODEL_ID, "model_revision": MODEL_REVISION,
        "dataset_path": str(Path(DATASET_PATH).resolve()),
        "dataset_sha256": hashlib.sha256(Path(DATASET_PATH).read_bytes()).hexdigest(),
        "training_seeds": TRAINING_SEEDS, "split_seed_policy": "same as training seed",
        "question_split_fractions": {"train": 0.8, "validation": 0.1, "test": 0.1},
        "lora_ranks": LORA_RANKS,
        "lora_target_suffixes": LORA_TARGET_SUFFIXES,
        "configurations": [
            {"rank": rank, "alpha": 2 * rank, "scaling": 2} for rank in LORA_RANKS
        ],
        "seed_splits": {
            str(seed): {
                "split_seed": data["split_seed"], "question_splits": data["question_splits"],
                "split_row_indices": {
                    name: [row["row_index"] for row in split] for name, split in data["splits"].items()
                },
                "row_counts": {name: len(split) for name, split in data["splits"].items()},
                "sequence_limit": data["sequence_limit"],
                "optimizer_steps_per_epoch": data["optimizer_steps_per_epoch"],
                "quarter_epoch_steps": data["quarter_epoch_steps"],
                "training_args_by_rank": {
                    str(rank): make_training_args(
                        configuration_dir(experiment_dir, seed, rank), compute_dtype, seed, data,
                    ).to_dict()
                    for rank in LORA_RANKS
                },
            }
            for seed, data in seed_data.items()
        },
        "lora_dropout": LORA_DROPOUT, "use_rslora": False,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
        "generation": {
            "do_sample": False, "enable_thinking": False,
            "max_new_tokens_floor": GENERATION_MIN_TOKEN_BUDGET,
            "tokens_per_criterion": GENERATION_TOKENS_PER_CRITERION,
        },
        "package_versions": {name: version(name) for name in (
            "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "numpy", "pandas", "scikit-learn",
        )},
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_version": torch.version.cuda,
    }
    (experiment_dir / "experiment.json").write_text(json.dumps(experiment_manifest, indent=2, default=str) + "\n")
    print("Experiment directory:", experiment_dir)
    for training_seed in TRAINING_SEEDS:
        for index, rank in enumerate(LORA_RANKS):
            print(f"Starting seed {training_seed}, r={rank}, alpha={2 * rank}, scaling=2")
            metrics, rows = run_configuration(
                rank, training_seed, experiment_dir, include_baseline=(index == 0),
            )
            all_metrics.extend(metrics)
            all_prediction_rows.extend(rows)
            if metrics:
                pd.DataFrame(all_metrics).to_csv(experiment_dir / "qwen35_27b_evaluation_comparison.csv", index=False)
    if RUN_TEST_EVALUATION:
        validate_seed_results(pd.DataFrame(all_metrics), TRAINING_SEEDS)
        evaluation_completed = True
else:
    print("Data checks complete. Set RUN_TRAINING=True to run the rank sweep across all configured seeds.")


Experiment directory: qwen35-27b-grading-qlora-inprojqkv-outproj-rank-multiseed/run-20261024T033024-090689Z
Starting seed 42, r=32, alpha=64, scaling=2


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Loaded in 4-bit: True | Compute dtype: torch.bfloat16
LoRA projection layers: 352
Targets by projection: {'q_proj': 16, 'k_proj': 16, 'v_proj': 16, 'o_proj': 16, 'gate_proj': 64, 'up_proj': 64, 'down_proj': 64, 'in_proj_qkv': 48, 'out_proj': 48}


trainable params: 200,278,016 || all params: 27,557,006,576 || trainable%: 0.7268


Truncating train dataset:   0%|          | 0/2980 [00:00<?, ? examples/s]

...

{'valid': True, 'raw_output': '[1, 0, 0]', 'scores': [1, 0, 0], 'error': None, 'recovered': False}


Starting seed 2026, r=40, alpha=80, scaling=2


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Loaded in 4-bit: True | Compute dtype: torch.bfloat16
LoRA projection layers: 352
Targets by projection: {'q_proj': 16, 'k_proj': 16, 'v_proj': 16, 'o_proj': 16, 'gate_proj': 64, 'up_proj': 64, 'down_proj': 64, 'in_proj_qkv': 48, 'out_proj': 48}


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 44.39 GiB of which 3.25 MiB is free. Including non-PyTorch memory, this process has 44.37 GiB memory in use. Of the allocated memory 43.69 GiB is allocated by PyTorch, and 170.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Inspect predictions

Preview 20 predictions with seed, rank, alpha, raw output, parsed decisions and final-score errors. Baseline rank/alpha are empty. Complete predictions are saved per seed and configuration.


In [ ]:
if evaluation_completed:
    final_score_predictions_df = pd.DataFrame(all_prediction_rows)
    display(final_score_predictions_df[[
        "seed", "rank", "alpha", "model", "row_index", "ground_truth", "prediction", "raw_output",
        "true_final_score", "predicted_final_score", "final_score_absolute_error", "valid", "recovered",
    ]].head(20))


## Compare ranks across seeds

Display per-seed results, equal-weight means, sample standard deviations (`ddof=1`) and QLoRA-minus-baseline differences for each rank. Across-seed variation includes both training randomness and changing question splits.

The concise table shows QWK, Pearson, RMSE, MAE and exact final-score accuracy. Highlight all tied winners using unrounded means; display four decimals and export full precision. Undefined values are skipped when averaging. Test results describe performance and do not select checkpoints or an upload.


In [ ]:
METRIC_COLUMNS = {
    "final_score_qwk": "QWK",
    "final_score_pearson": "Pearson",
    "final_score_rmse": "RMSE",
    "final_score_mae": "MAE",
    "final_score_exact_accuracy": "Accuracy",
}
HIGHER_IS_BETTER = ["QWK", "Pearson", "Accuracy"]
LOWER_IS_BETTER = ["RMSE", "MAE"]


def build_evaluation_tables(all_metrics):
    """Order all seed/configuration results and average evaluation metrics only."""
    results = pd.DataFrame(all_metrics)
    validate_seed_results(results, TRAINING_SEEDS)
    metric_columns = [column for column in results if column not in {"seed", "rank", "alpha", "model"}]
    model_order = ["Baseline", *(configuration_label(rank) for rank in LORA_RANKS)]
    row_order = pd.MultiIndex.from_product(
        [TRAINING_SEEDS, model_order],
        names=["seed", "model"],
    )
    ordered = results.set_index(["seed", "model"]).reindex(row_order)
    per_seed = ordered[metric_columns]
    grouped = per_seed.groupby(level="model", sort=False)
    averages = grouped.mean().reindex(model_order)
    standard_deviations = grouped.std(ddof=1).reindex(model_order)
    return per_seed, averages, standard_deviations


def display_metric_table(table, title):
    print(title)
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None,
        "display.float_format", "{:.4f}".format,
    ):
        display(table)


def metric_winners(table):
    """Return all tied winners per metric, omitting undefined values."""
    winners = {}
    for metric in table.columns:
        values = table[metric].dropna()
        best = values.max() if metric in HIGHER_IS_BETTER else values.min()
        winners[metric] = values.index[values.eq(best)].tolist()
    return winners


if evaluation_completed:
    results_df, average_results_df, standard_deviations_df = build_evaluation_tables(all_metrics)
    display_metric_table(results_df, "Table 1: Performance Across Testing Seeds")
    display_metric_table(average_results_df, "Table 2: Average Performance Across Testing Seeds")
    results_df.to_csv(experiment_dir / "per_seed_results.csv")
    average_results_df.to_csv(experiment_dir / "average_results.csv")
    standard_deviations_df.to_csv(experiment_dir / "across_seed_standard_deviations.csv")
    for seed in TRAINING_SEEDS:
        seed_results = results_df.xs(seed, level="seed")
        differences = seed_results.drop(index="Baseline").subtract(seed_results.loc["Baseline"], axis="columns")
        differences.to_csv(experiment_dir / f"seed_{seed}_comparison_to_baseline.csv")
    comparison_df = average_results_df[list(METRIC_COLUMNS)].rename(columns=METRIC_COLUMNS)
    comparison_std_df = standard_deviations_df[list(METRIC_COLUMNS)].rename(columns=METRIC_COLUMNS)
    print("Mean Qwen3.5-27B rank sweep (alpha/rank=2); highlighted cells are metric winners.")
    display(
        comparison_df.style.format("{:.4f}", na_rep="NaN")
        .highlight_max(subset=HIGHER_IS_BETTER, color="#d9ead3")
        .highlight_min(subset=LOWER_IS_BETTER, color="#d9ead3")
    )
    comparison_df.to_csv(experiment_dir / "qwen35_27b_comparison.csv")
    comparison_std_df.to_csv(experiment_dir / "qwen35_27b_comparison_std.csv")
    display_metric_table(comparison_std_df, "Sample standard deviations across seeds (ddof=1)")
    for metric, winners in metric_winners(comparison_df).items():
        print(f"Best {metric}: {', '.join(winners) if winners else 'undefined for all models'}")
    rank_differences_df = comparison_df.drop(index="Baseline").subtract(
        comparison_df.loc["Baseline"], axis="columns",
    )
    display_metric_table(rank_differences_df, "Mean differences from Baseline")
    rank_differences_df.to_csv(experiment_dir / "rank_comparison_to_baseline.csv")
    print("Complete metrics, validity rates, and raw predictions saved in:", experiment_dir)

else:
    print("Result tables require completed test evaluation for every configured seed and rank.")


## Optional upload

Upload is disabled by default. Choose `UPLOAD_SEED`, `UPLOAD_RANK` and `HF_REPO_ID` explicitly, then enable `RUN_HF_UPLOAD` after the full sweep and evaluation finish. Upload the selected adapter directory, excluding intermediate checkpoints. Hugging Face authentication and repository write access are required.


In [ ]:
if RUN_HF_UPLOAD:
    if not (RUN_TRAINING and RUN_TEST_EVALUATION):
        raise RuntimeError(
            "Set RUN_TRAINING=True and RUN_TEST_EVALUATION=True before uploading."
        )
    if not evaluation_completed:
        raise RuntimeError("Complete the held-out evaluation cells before uploading.")
    if UPLOAD_RANK not in LORA_RANKS:
        raise ValueError("UPLOAD_RANK must be one of LORA_RANKS.")
    if UPLOAD_SEED not in TRAINING_SEEDS:
        raise ValueError("UPLOAD_SEED must be one of TRAINING_SEEDS.")
    adapter_dir = configuration_dir(experiment_dir, UPLOAD_SEED, UPLOAD_RANK)
    if not (adapter_dir / "adapter_config.json").is_file():
        raise RuntimeError(f"Selected adapter is missing: {adapter_dir}")
    login()
    upload_result = upload_folder(
        folder_path=str(adapter_dir),
        repo_id=HF_REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint-*"],
        commit_message=f"Upload evaluated Qwen3.5-27B QLoRA +in_proj_qkv +out_proj r={UPLOAD_RANK} alpha={2 * UPLOAD_RANK} seed={UPLOAD_SEED} adapter",
    )
    print("Uploaded evaluated adapter to:", upload_result)
else:
    print("Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.")
